# 07 — Dedup fix, corrected transfer, and paper figures

The last notebook you need to run. It does three things:

1. **Fixes the dedup bug.** Notebook 06 dropped the corpus with fewer *rows* when it
   found a duplicate pair. That kept the Kaggle re-upload (19,431 rows, of which
   8,002 are duplicates) and discarded the clean Hannousse original. Every transfer
   number involving that corpus was computed on data with 41 per cent redundancy.
   This notebook keeps the corpus with more *unique* URLs instead.

2. **Recomputes the transfer matrix** on the corrected corpus set, with the same
   five seeds and the same statistics, and writes fresh CSVs.

3. **Generates the two figures the paper actually needs** --- the structural
   artefact plot across all four corpora, and the transfer heatmap that makes the
   inversion result visible at a glance.

The five images already on your Drive (`figure1_summary`, `shap_summary`,
`cost_analysis`, `calibration`) are from the earlier "we detect phishing" framing
and are not used by the current paper. They are left in place but ignored.

Run top to bottom. It reuses the corpora already downloaded to Drive, so there is
no new download and it finishes in a few minutes.

## Cell 1 — Environment and modules

In [1]:
CKPT_SRC = 'import json, time, hashlib, subprocess\nfrom pathlib import Path\nimport joblib\n\n\ndef _git_hash(repo="/content/phishing-detection"):\n    try:\n        return subprocess.check_output(\n            ["git", "-C", repo, "rev-parse", "--short", "HEAD"],\n            stderr=subprocess.DEVNULL).decode().strip()\n    except Exception:\n        return "nogit"\n\n\nclass Checkpoint:\n    # Resumable checkpointing for Colab. Survives runtime resets via Drive.\n\n    def __init__(self, root="/content/phishing-detection/checkpoints",\n                 run_id=None, verbose=True):\n        self.root = Path(root)\n        self.run_id = run_id or time.strftime("run_%Y%m%d_%H%M%S")\n        self.dir = self.root / self.run_id\n        self.dir.mkdir(parents=True, exist_ok=True)\n        self.verbose = verbose\n        self.manifest_path = self.dir / "manifest.json"\n        self.manifest = (json.loads(self.manifest_path.read_text())\n                         if self.manifest_path.exists() else {})\n\n    @staticmethod\n    def _key(params):\n        if params is None:\n            return "default"\n        blob = json.dumps(params, sort_keys=True, default=str)\n        return hashlib.md5(blob.encode()).hexdigest()[:10]\n\n    def _path(self, stage, params=None):\n        return self.dir / (stage + "__" + self._key(params) + ".joblib")\n\n    def exists(self, stage, params=None):\n        return self._path(stage, params).exists()\n\n    def save(self, stage, obj, params=None, meta=None):\n        p = self._path(stage, params)\n        joblib.dump(obj, p, compress=3)\n        size_mb = round(p.stat().st_size / 1e6, 3)\n        self.manifest[p.name] = {\n            "stage": stage, "params": params, "meta": meta or {},\n            "git_commit": _git_hash(),\n            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            "size_mb": size_mb,\n        }\n        self.manifest_path.write_text(\n            json.dumps(self.manifest, indent=2, default=str))\n        if self.verbose:\n            print("[ckpt] saved " + p.name + "  (" + str(size_mb) + " MB)")\n        return p\n\n    def load(self, stage, params=None):\n        p = self._path(stage, params)\n        if not p.exists():\n            raise FileNotFoundError("No checkpoint: " + str(p))\n        if self.verbose:\n            print("[ckpt] loaded " + p.name)\n        return joblib.load(p)\n\n    def cache(self, stage, fn, params=None, meta=None, force=False):\n        if self.exists(stage, params) and not force:\n            return self.load(stage, params)\n        if self.verbose:\n            print("[ckpt] computing " + stage + " ...")\n        out = fn()\n        self.save(stage, out, params=params, meta=meta)\n        return out\n\n    def summary(self):\n        import pandas as pd\n        if not self.manifest:\n            return pd.DataFrame()\n        return pd.DataFrame(self.manifest).T[\n            ["stage", "saved_at", "size_mb", "git_commit"]]\n\n    @classmethod\n    def latest_run(cls, root="/content/phishing-detection/checkpoints", **kw):\n        runs = sorted(p.name for p in Path(root).glob("run_*") if p.is_dir())\n        if not runs:\n            raise FileNotFoundError("No previous runs.")\n        return cls(root=root, run_id=runs[-1], **kw)\n'

URLFEAT_SRC = '"""Canonical URL-lexical features + split strategies.\n\nHardened against malformed URLs. Real corpora contain entries that make\nurllib.parse raise: non-numeric or out-of-range ports, unbalanced square\nbrackets that look like broken IPv6 literals, and embedded control\ncharacters. urlparse() is lazy about this - it raises on the .hostname\nand .port properties rather than at parse time - so every access is\nguarded here.\n"""\nimport re, math\nfrom urllib.parse import urlparse, SplitResult\nimport numpy as np\nimport pandas as pd\n\nSUSPICIOUS_WORDS = [\n    "login", "signin", "verify", "account", "update", "secure", "webscr",\n    "banking", "confirm", "password", "credential", "invoice", "payment",\n    "billing", "suspend", "unlock", "recover", "wallet", "support",\n]\nSHORTENERS = {\n    "bit.ly", "goo.gl", "tinyurl.com", "t.co", "ow.ly", "is.gd", "buff.ly",\n    "adf.ly", "bit.do", "cutt.ly", "rb.gy", "shorte.st", "rebrand.ly",\n}\nIPV4 = re.compile(r"^(?:\\d{1,3}\\.){3}\\d{1,3}$")\nHEX_HOST = re.compile(r"^0x[0-9a-f]+$", re.I)\nCTRL = re.compile(r"[\\x00-\\x1f\\x7f]")\n\n_EMPTY = SplitResult(scheme="", netloc="", path="", query="", fragment="")\n\n\ndef safe_parse(url):\n    """Parse a URL, never raising. Returns a SplitResult-like object.\n\n    Falls back through three stages: parse as given; strip control\n    characters and square brackets and retry; return an empty result.\n    """\n    s = str(url).strip()\n    if "://" not in s:\n        s = "http://" + s\n    try:\n        p = urlparse(s)\n        _ = p.netloc          # force any lazy validation we can\n        return p\n    except ValueError:\n        pass\n    cleaned = CTRL.sub("", s).replace("[", "").replace("]", "")\n    try:\n        return urlparse(cleaned)\n    except ValueError:\n        return _EMPTY\n\n\ndef safe_hostname(p):\n    """netloc-derived hostname, tolerating malformed authority sections."""\n    try:\n        h = p.hostname\n        if h is not None:\n            return h.lower()\n    except ValueError:\n        pass\n    netloc = getattr(p, "netloc", "") or ""\n    netloc = netloc.rsplit("@", 1)[-1]          # drop userinfo\n    netloc = CTRL.sub("", netloc).replace("[", "").replace("]", "")\n    host = netloc.split(":", 1)[0]\n    return host.lower()\n\n\ndef safe_port(p):\n    """Port if it is a valid integer in range, else None. Never raises."""\n    try:\n        return p.port\n    except ValueError:\n        return None\n\n\ndef is_malformed(url):\n    """True if the URL needed the fallback path. Report this rate."""\n    s = str(url).strip()\n    if "://" not in s:\n        s = "http://" + s\n    try:\n        p = urlparse(s)\n        _ = p.hostname\n        _ = p.port\n        return False\n    except ValueError:\n        return True\n\n\ndef _entropy(s):\n    if not s:\n        return 0.0\n    counts = {}\n    for ch in s:\n        counts[ch] = counts.get(ch, 0) + 1\n    n = len(s)\n    return -sum((c / n) * math.log2(c / n) for c in counts.values())\n\n\ndef canonical_url_features(url, extractor=None):\n    """~32 lexical features computable from a URL alone, identically for any source."""\n    url = str(url).strip()\n    if "://" not in url:\n        url = "http://" + url\n\n    p = safe_parse(url)\n    host = safe_hostname(p)\n    path = getattr(p, "path", "") or ""\n    query = getattr(p, "query", "") or ""\n\n    subdomain = domain = suffix = ""\n    if extractor is not None:\n        try:\n            ext = extractor(url)\n            subdomain, domain, suffix = ext.subdomain, ext.domain, ext.suffix\n        except Exception:\n            extractor = None\n    if not (domain or suffix):\n        parts = [x for x in host.split(".") if x]\n        suffix = parts[-1] if len(parts) > 1 else ""\n        domain = parts[-2] if len(parts) > 2 else (parts[0] if parts else "")\n        subdomain = ".".join(parts[:-2]) if len(parts) > 2 else ""\n\n    registrable = f"{domain}.{suffix}".strip(".")\n    digits = sum(ch.isdigit() for ch in url)\n    letters = sum(ch.isalpha() for ch in url)\n\n    return {\n        "url_length": len(url),\n        "host_length": len(host),\n        "path_length": len(path),\n        "query_length": len(query),\n        "n_dots": url.count("."),\n        "n_hyphens": url.count("-"),\n        "n_underscores": url.count("_"),\n        "n_slashes": path.count("/"),\n        "n_question": url.count("?"),\n        "n_equals": url.count("="),\n        "n_at": url.count("@"),\n        "n_ampersand": url.count("&"),\n        "n_percent": url.count("%"),\n        "n_digits": digits,\n        "n_letters": letters,\n        "digit_ratio": digits / max(len(url), 1),\n        "n_subdomains": len([s for s in subdomain.split(".") if s]),\n        "subdomain_length": len(subdomain),\n        "domain_length": len(domain),\n        "tld_length": len(suffix),\n        "is_ip_host": int(bool(IPV4.match(host))),\n        "is_hex_host": int(bool(HEX_HOST.match(host))),\n        "is_https": int(str(getattr(p, "scheme", "")).lower() == "https"),\n        "has_port": int(safe_port(p) is not None),\n        "is_shortener": int(registrable in SHORTENERS),\n        "has_punycode": int("xn--" in host),\n        "has_double_slash_path": int("//" in path),\n        "n_suspicious_words": sum(w in url.lower() for w in SUSPICIOUS_WORDS),\n        "url_entropy": _entropy(url),\n        "host_entropy": _entropy(host),\n        "longest_token_path": max([len(t) for t in re.split(r"[/\\-_.]", path) if t] or [0]),\n        "has_https_token_in_host": int("https" in host),\n    }\n\n\ndef build_canonical_matrix(urls, use_tldextract=True):\n    extractor = None\n    if use_tldextract:\n        extractor = get_extractor()\n    rows = [canonical_url_features(u, extractor) for u in urls]\n    return pd.DataFrame(rows, index=getattr(urls, "index", None))\n\n\n_DEFAULT_EXTRACTOR = None\n\n\ndef get_extractor():\n    """Cached tldextract instance using the bundled PSL snapshot (no network)."""\n    global _DEFAULT_EXTRACTOR\n    if _DEFAULT_EXTRACTOR is None:\n        try:\n            import tldextract\n            _DEFAULT_EXTRACTOR = tldextract.TLDExtract(suffix_list_urls=())\n        except Exception:\n            _DEFAULT_EXTRACTOR = False\n    return _DEFAULT_EXTRACTOR or None\n\n\ndef registrable_domain(url, extractor="auto"):\n    """eTLD+1. Returns the bare IP for IP-literal hosts. Never raises."""\n    s = str(url).strip()\n    if "://" not in s:\n        s = "http://" + s\n    host = safe_hostname(safe_parse(s))\n    if not host:\n        return ""\n    if IPV4.match(host):\n        return host\n    if extractor == "auto":\n        extractor = get_extractor()\n    if extractor is not None:\n        try:\n            e = extractor(s)\n            reg = f"{e.domain}.{e.suffix}".strip(".").strip("[]")\n            if reg:\n                return reg\n        except Exception:\n            pass\n    parts = [x for x in host.strip("[]").split(".") if x]\n    return ".".join(parts[-2:]) if len(parts) >= 2 else host.strip("[]")\n\n\n# ------------------------------------------------------------------ splits\ndef grouped_split(X, y, groups, test_size=0.2, seed=42):\n    """No registrable domain appears in both train and test."""\n    from sklearn.model_selection import GroupShuffleSplit\n    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)\n    tr, te = next(gss.split(X, y, groups))\n    return tr, te\n\n\ndef stratified_split(X, y, test_size=0.2, seed=42):\n    from sklearn.model_selection import train_test_split\n    idx = np.arange(len(y))\n    return train_test_split(idx, test_size=test_size, random_state=seed, stratify=y)\n\n\ndef rebalance_to_base_rate(y, target_rate, seed=42, index=None):\n    """Downsample the phishing class to a realistic prevalence."""\n    rng = np.random.default_rng(seed)\n    idx = np.arange(len(y)) if index is None else np.asarray(index)\n    pos = idx[np.asarray(y) == 1]\n    neg = idx[np.asarray(y) == 0]\n    n_pos = int(round(target_rate * len(neg) / (1 - target_rate)))\n    n_pos = min(n_pos, len(pos))\n    keep = np.concatenate([neg, rng.choice(pos, size=n_pos, replace=False)])\n    rng.shuffle(keep)\n    return keep\n'

STAT_SRC = '"""Inference for transfer experiments: DeLong, bootstrap CIs, corrections."""\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\n\n# ------------------------------------------------------------------ DeLong\ndef _midrank(x):\n    J = np.argsort(x)\n    Z = x[J]\n    N = len(x)\n    T = np.zeros(N, dtype=float)\n    i = 0\n    while i < N:\n        j = i\n        while j < N and Z[j] == Z[i]:\n            j += 1\n        T[i:j] = 0.5 * (i + j - 1) + 1\n        i = j\n    T2 = np.empty(N, dtype=float)\n    T2[J] = T\n    return T2\n\n\ndef _structural_components(scores, n_pos, n_neg):\n    """V10 (positives) and V01 (negatives) components for each predictor."""\n    k = scores.shape[0]\n    tx = np.empty([k, n_pos]); ty = np.empty([k, n_neg]); tz = np.empty([k, n_pos + n_neg])\n    for r in range(k):\n        pos = scores[r, :n_pos]\n        neg = scores[r, n_pos:]\n        tx[r, :] = _midrank(pos)\n        ty[r, :] = _midrank(neg)\n        tz[r, :] = _midrank(scores[r, :])\n    aucs = (tz[:, :n_pos].sum(axis=1) / n_pos - (n_pos + 1) / 2) / n_neg\n    v01 = (tz[:, :n_pos] - tx) / n_neg\n    v10 = 1 - (tz[:, n_pos:] - ty) / n_pos\n    return aucs, v01, v10\n\n\ndef delong_test(y_true, scores_a, scores_b):\n    """Paired DeLong test for two AUCs on the same sample.\n\n    Returns (auc_a, auc_b, z, p). Two-sided.\n    """\n    y = np.asarray(y_true).astype(int)\n    order = np.argsort(-y, kind="mergesort")   # positives first\n    y = y[order]\n    a = np.asarray(scores_a)[order]\n    b = np.asarray(scores_b)[order]\n    n_pos = int(y.sum()); n_neg = len(y) - n_pos\n    if n_pos == 0 or n_neg == 0:\n        return np.nan, np.nan, np.nan, np.nan\n\n    scores = np.vstack([a, b])\n    aucs, v01, v10 = _structural_components(scores, n_pos, n_neg)\n    s01 = np.cov(v01); s10 = np.cov(v10)\n    s = s01 / n_pos + s10 / n_neg\n    var = s[0, 0] + s[1, 1] - 2 * s[0, 1]\n    if var <= 0:\n        return float(aucs[0]), float(aucs[1]), np.nan, np.nan\n    z = (aucs[0] - aucs[1]) / np.sqrt(var)\n    p = 2 * stats.norm.sf(abs(z))\n    return float(aucs[0]), float(aucs[1]), float(z), float(p)\n\n\ndef delong_ci(y_true, scores, alpha=0.05):\n    """DeLong analytic confidence interval for a single AUC."""\n    y = np.asarray(y_true).astype(int)\n    order = np.argsort(-y, kind="mergesort")\n    y = y[order]; s = np.asarray(scores)[order]\n    n_pos = int(y.sum()); n_neg = len(y) - n_pos\n    if n_pos == 0 or n_neg == 0:\n        return np.nan, np.nan, np.nan\n    aucs, v01, v10 = _structural_components(s.reshape(1, -1), n_pos, n_neg)\n    var = np.var(v01, ddof=1) / n_pos + np.var(v10, ddof=1) / n_neg\n    se = np.sqrt(var)\n    zc = stats.norm.ppf(1 - alpha / 2)\n    auc = float(aucs[0])\n    return auc, max(0.0, auc - zc * se), min(1.0, auc + zc * se)\n\n\n# --------------------------------------------------------------- bootstrap\ndef bootstrap_ci(y_true, scores, metric_fn, n_boot=2000, alpha=0.05, seed=42):\n    """Stratified percentile bootstrap CI for any metric(y, s)."""\n    rng = np.random.default_rng(seed)\n    y = np.asarray(y_true); s = np.asarray(scores)\n    pos = np.flatnonzero(y == 1); neg = np.flatnonzero(y == 0)\n    if len(pos) < 2 or len(neg) < 2:\n        return np.nan, np.nan, np.nan\n    vals = np.empty(n_boot)\n    for i in range(n_boot):\n        idx = np.concatenate([rng.choice(pos, len(pos), replace=True),\n                              rng.choice(neg, len(neg), replace=True)])\n        try:\n            vals[i] = metric_fn(y[idx], s[idx])\n        except Exception:\n            vals[i] = np.nan\n    vals = vals[~np.isnan(vals)]\n    point = metric_fn(y, s)\n    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])\n    return float(point), float(lo), float(hi)\n\n\n# ------------------------------------------------------- permutation tests\ndef permutation_gap_test(within_vals, cross_vals, n_perm=10000, seed=42):\n    """Is the within-vs-cross gap larger than chance reassignment would give?"""\n    rng = np.random.default_rng(seed)\n    a = np.asarray(within_vals, dtype=float); b = np.asarray(cross_vals, dtype=float)\n    obs = a.mean() - b.mean()\n    pool = np.concatenate([a, b]); n = len(a)\n    count = 0\n    for _ in range(n_perm):\n        rng.shuffle(pool)\n        if (pool[:n].mean() - pool[n:].mean()) >= obs:\n            count += 1\n    return float(obs), (count + 1) / (n_perm + 1)\n\n\ndef cliffs_delta(a, b):\n    """Non-parametric effect size. |d|: .147 small, .33 medium, .474 large."""\n    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)\n    gt = sum((x > y) for x in a for y in b)\n    lt = sum((x < y) for x in a for y in b)\n    return (gt - lt) / (len(a) * len(b))\n\n\n# --------------------------------------------- multiple comparison control\ndef holm_bonferroni(pvals, alpha=0.05):\n    """Holm step-down. Returns (adjusted p, reject) in the original order."""\n    p = np.asarray(pvals, dtype=float)\n    ok = ~np.isnan(p)\n    out = np.full(len(p), np.nan); rej = np.zeros(len(p), dtype=bool)\n    idx = np.flatnonzero(ok)\n    if len(idx) == 0:\n        return out, rej\n    order = idx[np.argsort(p[idx])]\n    m = len(order)\n    running = 0.0\n    for rank, i in enumerate(order):\n        adj = min(1.0, (m - rank) * p[i])\n        running = max(running, adj)     # enforce monotonicity\n        out[i] = running\n        rej[i] = running <= alpha\n    return out, rej\n'

from google.colab import drive
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")

drive.mount('/content/drive')
PROJECT   = "phishing-detection"
REPO_DIR  = Path("/content") / PROJECT
DRIVE_DIR = Path("/content/drive/MyDrive/research") / PROJECT

for d in ["src", "src/features", "src/evaluation", "src/utils",
          "reports/tables", "reports/figures"]:
    (REPO_DIR / d).mkdir(parents=True, exist_ok=True)
for d in ["src", "src/features", "src/evaluation", "src/utils"]:
    (REPO_DIR / d / "__init__.py").touch()
for name in ["data", "checkpoints", "models", "logs"]:
    link = REPO_DIR / name
    if link.is_symlink(): link.unlink()
    elif link.exists(): raise RuntimeError(f"{link} exists and is not a symlink")
    link.symlink_to(DRIVE_DIR / name)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

(REPO_DIR / "src" / "utils" / "checkpoint.py").write_text(CKPT_SRC)
(REPO_DIR / "src" / "features" / "urlfeat.py").write_text(URLFEAT_SRC)
(REPO_DIR / "src" / "evaluation" / "stats.py").write_text(STAT_SRC)

import importlib
import src.features.urlfeat, src.evaluation.stats, src.utils.checkpoint
importlib.reload(src.features.urlfeat); importlib.reload(src.evaluation.stats)
importlib.reload(src.utils.checkpoint)
import src.features.urlfeat as uf
import src.evaluation.stats as ST
from src.utils.checkpoint import Checkpoint

RAW = REPO_DIR / "data" / "raw"
TABLES = REPO_DIR / "reports" / "tables"
FIGS   = REPO_DIR / "reports" / "figures"
print("modules loaded; raw ->", RAW.resolve())

Mounted at /content/drive
modules loaded; raw -> /content/drive/MyDrive/research/phishing-detection/data/raw


## Cell 2 — Reload corpora

Same five URL-bearing corpora as notebook 06, loaded from the CSVs already on
Drive. No download.

In [2]:
import pandas as pd, numpy as np

LOADERS = {
    "phiusiil":          ("phiusiil/phiusiil.csv",              "URL",  "label",       0),
    "hannousse":         ("hannousse/hannousse.csv",            "url",  "status",      "phishing"),
    "kaggle_malicious":  ("kaggle_malicious/kaggle_malicious_urls.csv",   "url", "label", 1),
    "kaggle_linkphish":  ("kaggle_linkphish/kaggle_linkphish_urls.csv",   "url", "label", 1),
    "mendeley_phishurl": ("mendeley_phishurl/mendeley_phishurl_urls.csv", "url", "label", 1),
}

CAP = 60000
rng = np.random.default_rng(42)
CORPORA = {}
for key, (rel, ucol, lcol, phish) in LOADERS.items():
    p = RAW / rel
    if not p.exists():
        print(f"[{key}] missing at {p}"); continue
    d = pd.read_csv(p, low_memory=False)
    uc = [col for col in d.columns if col.lower() == ucol.lower()][0]
    lc = [col for col in d.columns if col.lower() == lcol.lower()][0]
    lab = ((d[lc].astype(str).str.lower() == str(phish).lower()).astype(int)
           if isinstance(phish, str) else (d[lc] == phish).astype(int))
    df = pd.DataFrame({"url": d[uc].astype(str), "label": lab})
    if len(df) > CAP:
        idx = np.sort(rng.choice(len(df), CAP, replace=False))
        df = df.iloc[idx].reset_index(drop=True)
    CORPORA[key] = df
    print(f"[{key}] {len(df):,} rows")

print(f"\n{len(CORPORA)} corpora loaded")

[phiusiil] 60,000 rows
[hannousse] 11,430 rows
[kaggle_malicious] 60,000 rows
[kaggle_linkphish] 19,431 rows
[mendeley_phishurl] 60,000 rows

5 corpora loaded


## Cell 3 — Dedup, corrected

The fix. When two corpora share essentially all their URLs, keep the one with more
**unique normalised** URLs, not more rows. That retains the clean original and drops
the duplicated re-upload --- the opposite of what notebook 06 did.

In [3]:
def normalise(u):
    s = str(u).strip().lower()
    for pre in ("https://", "http://"):
        if s.startswith(pre):
            s = s[len(pre):]; break
    if s.startswith("www."):
        s = s[4:]
    return s.rstrip("/")

norm_sets = {k: set(v["url"].map(normalise)) for k, v in CORPORA.items()}

ALL = list(CORPORA)
overlap_rows, flagged = [], []
for i, a in enumerate(ALL):
    for b in ALL[i+1:]:
        inter = len(norm_sets[a] & norm_sets[b])
        mn = min(len(norm_sets[a]), len(norm_sets[b]))
        cont = inter / mn if mn else 0.0
        overlap_rows.append({"corpus_a": a, "corpus_b": b, "shared": inter,
                             "containment": round(cont, 4)})
        if cont > 0.5:
            flagged.append((a, b))

overlap = pd.DataFrame(overlap_rows).sort_values("containment", ascending=False)
print(overlap.to_string(index=False))

DROP = set()
for a, b in flagged:
    # keep the corpus with MORE unique normalised URLs
    loser = a if len(norm_sets[a]) < len(norm_sets[b]) else b
    winner = b if loser == a else a
    DROP.add(loser)
    print(f"\nduplicate pair {a} / {b}:")
    print(f"  unique normalised URLs: {a}={len(norm_sets[a]):,}, {b}={len(norm_sets[b]):,}")
    print(f"  -> keeping {winner} (more unique), dropping {loser}")

if not flagged:
    print("\nno duplicate pairs found")

KEEP = [k for k in CORPORA if k not in DROP]
print(f"\ncorpora carried forward: {KEEP}")

        corpus_a          corpus_b  shared  containment
       hannousse  kaggle_linkphish   11070       1.0000
kaggle_malicious mendeley_phishurl    4618       0.0773
kaggle_malicious  kaggle_linkphish     347       0.0313
       hannousse  kaggle_malicious     347       0.0313
       hannousse mendeley_phishurl     153       0.0138
kaggle_linkphish mendeley_phishurl     153       0.0138
        phiusiil  kaggle_linkphish      62       0.0056
        phiusiil         hannousse      62       0.0056
        phiusiil mendeley_phishurl     222       0.0037
        phiusiil  kaggle_malicious      34       0.0006

duplicate pair hannousse / kaggle_linkphish:
  unique normalised URLs: hannousse=11,070, kaggle_linkphish=11,070
  -> keeping hannousse (more unique), dropping kaggle_linkphish

corpora carried forward: ['phiusiil', 'hannousse', 'kaggle_malicious', 'mendeley_phishurl']


## Cell 4 — Canonical feature matrices on the corrected set

In [4]:
canon = {}
for key in KEEP:
    df = CORPORA[key]
    urls = df["url"].astype(str)
    X = uf.build_canonical_matrix(urls)
    dom = urls.map(uf.registrable_domain)
    canon[key] = {"X": X.reset_index(drop=True),
                  "y": df["label"].reset_index(drop=True),
                  "domain": dom.reset_index(drop=True)}
    print(f"{key:20s} X={str(X.shape):>13s}  domains={dom.nunique():>7,}  "
          f"phishing_rate={df['label'].mean():.4f}")

KEYS = list(canon)

phiusiil             X=  (60000, 32)  domains= 42,159  phishing_rate=0.4267
hannousse            X=  (11430, 32)  domains=  6,058  phishing_rate=0.5000
kaggle_malicious     X=  (60000, 32)  domains= 24,501  phishing_rate=0.1805
mendeley_phishurl    X=  (60000, 32)  domains= 25,130  phishing_rate=0.3155


## Cell 5 — Recompute the transfer matrix

Same protocol as before: every ordered pair, five seeds, XGBoost, domain-grouped
splits on the diagonal.

In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef

SEEDS = (42, 43, 44, 45, 46)

def make_model(seed):
    return XGBClassifier(n_estimators=400, max_depth=7, learning_rate=0.1,
                         subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
                         eval_metric="logloss", random_state=seed, verbosity=0)

records, raw_scores = [], {}
for src in KEYS:
    for dst in KEYS:
        for seed in SEEDS:
            Xs, ys = canon[src]["X"], canon[src]["y"]
            if src == dst:
                tr, te = uf.grouped_split(Xs, ys, canon[src]["domain"], seed=seed)
                Xtr, ytr, Xte, yte = Xs.iloc[tr], ys.iloc[tr], Xs.iloc[te], ys.iloc[te]
            else:
                Xd, yd = canon[dst]["X"], canon[dst]["y"]
                cols = [col for col in Xs.columns if col in Xd.columns]
                Xtr, ytr, Xte, yte = Xs[cols], ys, Xd[cols], yd
            yte_arr = np.asarray(yte)
            if len(np.unique(yte_arr)) < 2 or len(np.unique(np.asarray(ytr))) < 2:
                print(f"  [skip] {src}->{dst} seed={seed}: single-class partition")
                continue
            m = make_model(seed).fit(Xtr, ytr)
            s = m.predict_proba(Xte)[:, 1]
            raw_scores[(src, dst, seed)] = (yte_arr, s)
            records.append({"train": src, "test": dst, "seed": seed,
                            "kind": "within" if src == dst else "cross",
                            "roc_auc": roc_auc_score(yte, s),
                            "f1": f1_score(yte, (s >= 0.5).astype(int), zero_division=0),
                            "mcc": matthews_corrcoef(yte, (s >= 0.5).astype(int))})
        print(f"  {src} -> {dst}: done")

transfer = pd.DataFrame(records)
transfer.to_csv(TABLES / "transfer_matrix.csv", index=False)
piv = transfer.pivot_table(index="train", columns="test", values="roc_auc").round(4)
print("\nCorrected mean ROC-AUC (rows = train, cols = test):")
print(piv.to_string())

  phiusiil -> phiusiil: done
  phiusiil -> hannousse: done
  phiusiil -> kaggle_malicious: done
  phiusiil -> mendeley_phishurl: done
  hannousse -> phiusiil: done
  hannousse -> hannousse: done
  hannousse -> kaggle_malicious: done
  hannousse -> mendeley_phishurl: done
  kaggle_malicious -> phiusiil: done
  kaggle_malicious -> hannousse: done
  kaggle_malicious -> kaggle_malicious: done
  kaggle_malicious -> mendeley_phishurl: done
  mendeley_phishurl -> phiusiil: done
  mendeley_phishurl -> hannousse: done
  mendeley_phishurl -> kaggle_malicious: done
  mendeley_phishurl -> mendeley_phishurl: done

Corrected mean ROC-AUC (rows = train, cols = test):
test               hannousse  kaggle_malicious  mendeley_phishurl  phiusiil
train                                                                      
hannousse             0.9437            0.5561             0.9400    0.8603
kaggle_malicious      0.4618            0.9426             0.2020    0.3667
mendeley_phishurl     0.7823       

## Cell 6 — Corrected statistics

In [6]:
within = transfer.loc[transfer.kind == "within", "roc_auc"].values
cross  = transfer.loc[transfer.kind == "cross",  "roc_auc"].values
within = within[~np.isnan(within)]; cross = cross[~np.isnan(cross)]

obs, p_perm = ST.permutation_gap_test(within, cross, n_perm=10000)
delta = ST.cliffs_delta(within, cross)

print(f"within  n={len(within):3d}  mean={within.mean():.4f}  sd={within.std(ddof=1):.4f}")
print(f"cross   n={len(cross):3d}  mean={cross.mean():.4f}  sd={cross.std(ddof=1):.4f}")
print(f"gap {obs:.4f}   permutation p={p_perm:.5f} (report as p<0.001)   "
      f"Cliff delta={delta:.4f}")

n_inv = int((cross < 0.5).sum())
print(f"\ninverted transfer cells (AUC<0.5): {n_inv} of {len(cross)}")
print(f"lowest transfer AUC: {cross.min():.4f}")

summary = pd.DataFrame([{
    "within_mean": round(within.mean(), 4), "within_sd": round(within.std(ddof=1), 4),
    "cross_mean": round(cross.mean(), 4), "cross_sd": round(cross.std(ddof=1), 4),
    "gap": round(obs, 4), "cliffs_delta": round(delta, 4),
    "n_inverted": n_inv, "min_transfer_auc": round(cross.min(), 4),
    "n_corpora": len(KEYS)}])
summary.to_csv(TABLES / "transfer_summary.csv", index=False)
print("\nsaved -> transfer_summary.csv  (send me this one)")

within  n= 20  mean=0.9700  sd=0.0280
cross   n= 60  mean=0.6038  sd=0.2867
gap 0.3662   permutation p=0.00010 (report as p<0.001)   Cliff delta=0.8883

inverted transfer cells (AUC<0.5): 25 of 60
lowest transfer AUC: 0.1898

saved -> transfer_summary.csv  (send me this one)


## Cell 7 — Confidence intervals and asymmetry (corrected)

In [7]:
ci_rows = []
for (src, dst, seed), (y, s) in raw_scores.items():
    if seed != SEEDS[0]:
        continue
    pt, lo, hi = ST.bootstrap_ci(y, s, roc_auc_score, n_boot=2000, seed=seed)
    ci_rows.append({"train": src, "test": dst,
                    "kind": "within" if src == dst else "cross",
                    "roc_auc": round(pt, 4),
                    "ci_lo": round(lo, 4), "ci_hi": round(hi, 4)})
cis = pd.DataFrame(ci_rows).sort_values(["kind", "roc_auc"])
cis.to_csv(TABLES / "transfer_cis.csv", index=False)
print(cis.to_string(index=False))

from scipy import stats as sps
pairs, pvals = [], []
for i, a in enumerate(KEYS):
    for b in KEYS[i+1:]:
        ab = transfer[(transfer.train == a) & (transfer.test == b)]["roc_auc"].values
        ba = transfer[(transfer.train == b) & (transfer.test == a)]["roc_auc"].values
        if len(ab) == 0 or len(ba) == 0:
            continue
        try:
            p = float(sps.mannwhitneyu(ab, ba, alternative="two-sided").pvalue)
        except Exception:
            p = np.nan
        pairs.append({"corpus_a": a, "corpus_b": b,
                      "auc_a_to_b": round(ab.mean(), 4), "auc_b_to_a": round(ba.mean(), 4),
                      "asymmetry": round(abs(ab.mean() - ba.mean()), 4), "p_raw": p})
        pvals.append(p)
asym = pd.DataFrame(pairs)
if len(asym):
    adj, rej = ST.holm_bonferroni(asym["p_raw"].values)
    asym["p_holm"] = np.round(adj, 5); asym["significant"] = rej
    asym = asym.sort_values("asymmetry", ascending=False)
    asym.to_csv(TABLES / "asymmetry.csv", index=False)
    print("\n" + asym.to_string(index=False))

            train              test   kind  roc_auc  ci_lo  ci_hi
 kaggle_malicious mendeley_phishurl  cross   0.2004 0.1960 0.2049
mendeley_phishurl  kaggle_malicious  cross   0.2291 0.2240 0.2343
         phiusiil  kaggle_malicious  cross   0.2467 0.2411 0.2520
 kaggle_malicious          phiusiil  cross   0.3998 0.3948 0.4050
 kaggle_malicious         hannousse  cross   0.4596 0.4493 0.4706
        hannousse  kaggle_malicious  cross   0.5611 0.5553 0.5669
         phiusiil         hannousse  cross   0.6936 0.6834 0.7029
mendeley_phishurl         hannousse  cross   0.7802 0.7717 0.7885
        hannousse          phiusiil  cross   0.8720 0.8689 0.8752
         phiusiil mendeley_phishurl  cross   0.9210 0.9181 0.9238
        hannousse mendeley_phishurl  cross   0.9398 0.9378 0.9418
mendeley_phishurl          phiusiil  cross   0.9787 0.9771 0.9802
 kaggle_malicious  kaggle_malicious within   0.9363 0.9294 0.9432
        hannousse         hannousse within   0.9463 0.9379 0.9545
mendeley_p

## Cell 8 — Figure 1: the structural artefact

Path-length distribution by class, one panel per corpus. Shows PhiUSIIL's
degenerate legitimate class (a single spike at zero) against corpora where both
classes have paths.

In [ ]:
from urllib.parse import urlparse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.25, "savefig.dpi": 300})

def path_len(u):
    return len(getattr(uf.safe_parse(u), "path", "") or "")

n = len(KEYS)
ncol = min(n, 4)
fig, axes = plt.subplots(1, ncol, figsize=(3.4 * ncol, 3.0), sharey=True)
axes = np.atleast_1d(axes)
bins = np.arange(0, 81, 4)

for ax, key in zip(axes, KEYS):
    df = CORPORA[key]
    pl = df["url"].map(path_len)
    y = df["label"]
    ax.hist(np.clip(pl[y == 0], 0, 80), bins=bins, alpha=0.75, density=True,
            color="#4C72B0", label="Legitimate")
    ax.hist(np.clip(pl[y == 1], 0, 80), bins=bins, alpha=0.6, density=True,
            color="#C44E52", label="Phishing")
    ax.set_title(key, fontsize=9)
    ax.set_xlabel("URL path length")
    frac0 = (pl[y == 0] == 0).mean()
    if frac0 > 0.98:
        ax.annotate("legit class:\nall path length 0", xy=(2, ax.get_ylim()[1]*0.7),
                    fontsize=6.5, color="#4C72B0")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIGS / "fig_artefact_4corpora.pdf", bbox_inches="tight")
fig.savefig(FIGS / "fig_artefact_4corpora.png", bbox_inches="tight")
plt.close(fig)
print("saved -> fig_artefact_4corpora.pdf / .png")

## Cell 9 — Figure 2: transfer heatmap

The 4x4 matrix as a colour grid. Green is good transfer, red is failure; cells
below 0.5 (systematic inversion) are outlined. This is the figure that makes the
central finding visible in one glance.

In [ ]:
piv = transfer.pivot_table(index="train", columns="test", values="roc_auc")
piv = piv.reindex(index=sorted(piv.index), columns=sorted(piv.columns))

fig, ax = plt.subplots(figsize=(6.4, 5.4))
im = ax.imshow(piv.values, cmap="RdYlGn", vmin=0.0, vmax=1.0, aspect="equal")
ax.set_xticks(range(len(piv.columns))); ax.set_yticks(range(len(piv.index)))
ax.set_xticklabels(piv.columns, rotation=35, ha="right", fontsize=8)
ax.set_yticklabels(piv.index, fontsize=8)

for i in range(len(piv.index)):
    for j in range(len(piv.columns)):
        v = piv.values[i, j]
        if np.isnan(v):
            continue
        ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=9,
                fontweight="bold" if v < 0.5 else "normal")
        if v < 0.5:
            ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False,
                                       edgecolor="blue", lw=2.5))

ax.set_xlabel("Evaluation corpus"); ax.set_ylabel("Training corpus")
ax.set_title("Cross-corpus transfer (ROC-AUC)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="ROC-AUC")
fig.tight_layout()
fig.savefig(FIGS / "fig_transfer_heatmap.pdf", bbox_inches="tight")
fig.savefig(FIGS / "fig_transfer_heatmap.png", bbox_inches="tight")
plt.close(fig)

n_inv = int((piv.values[~np.eye(len(piv), dtype=bool)] < 0.5).sum())
print(f"saved -> fig_transfer_heatmap.pdf / .png  ({n_inv} inverted cells outlined)")

## Cell 10 — Bundle the outputs

In [ ]:
import shutil
bundle = REPO_DIR / "reports" / "paper_v3_inputs"
if bundle.exists():
    shutil.rmtree(bundle)
(bundle / "tables").mkdir(parents=True)
(bundle / "figures").mkdir(parents=True)

for name in ["transfer_matrix.csv", "transfer_cis.csv", "asymmetry.csv",
             "transfer_summary.csv", "structural_audit.csv", "corpus_overlap.csv",
             "data_quality.csv"]:
    src = TABLES / name
    if src.exists():
        shutil.copy2(src, bundle / "tables" / name)

for name in ["fig_artefact_4corpora.pdf", "fig_artefact_4corpora.png",
             "fig_transfer_heatmap.pdf", "fig_transfer_heatmap.png"]:
    src = FIGS / name
    if src.exists():
        shutil.copy2(src, bundle / "figures" / name)

archive = shutil.make_archive(str(DRIVE_DIR / "paper_v3_inputs"), "zip", bundle)
print("bundle written to Drive:")
print(" ", archive)
print("\nDownload paper_v3_inputs.zip from your Drive and send it to me.")
print("Contents:")
for f in sorted(bundle.rglob("*")):
    if f.is_file():
        print("  ", f.relative_to(bundle))

---
### After this runs

Download `paper_v3_inputs.zip` from your Drive root and send it. It has the
corrected transfer numbers and both figures. I will then update the manuscript so
every number matches the deduplicated run, drop in the two figures, and the results
section is final.

That leaves only the parts that are yours: the related-work citations, contacting
the PhiUSIIL authors, and the venue choice.